# MADT7202 — Pokémon TCG Real Decklist Network v3.1 FIXED

**Business question:** การ์ดใบใดควรอยู่ใน Core Watchlist ของร้าน และคู่การ์ดใดควรตรวจยอดคงเหลือพร้อมกันก่อนออกคำสั่งซื้อ (PO)

เวอร์ชันนี้ตั้งใจวาด **2 กราฟ** เพื่อแยก “ภาพของข้อมูลดิบ” ออกจาก “ภาพที่ใช้ตัดสินใจทางธุรกิจ”

- **Graph A — Pre-pruning Network:** แสดงทุก card co-occurrence หลัง technical cleaning เพื่อให้เห็นความหนาแน่น/noise ของ network ก่อนใส่ business criteria
- **Graph B — Actionable Stock Network:** ใช้เกณฑ์ที่กำหนดไว้ล่วงหน้าเพื่อตัด noise แล้วค่อยสร้าง network สำหรับ Core Watchlist และ Pair-check

> สำคัญ: Metric เช่น centrality คำนวณบน **Analysis Graph** ก่อน แล้วจึงลดรูปอีกครั้งเพื่อทำ **Display Backbone** สำหรับ A4 เท่านั้น


## ทำไมใช้เกณฑ์เหล่านี้ใน Graph B?

เกณฑ์ทั้งหมดถูกกำหนด **ก่อน ranking** เพื่อไม่ไล่ปรับค่าให้ได้ชื่อการ์ดที่ต้องการ

| เกณฑ์ | ค่า | เหตุผล |
|---|---:|---|
| ตัด Basic Energy | ตัดทั้งหมด | เป็น generic component ที่เชื่อมกว้างและไม่ช่วยแยก stock signal |
| Card prevalence | ≥ 10% | Core Watchlist ควรสะท้อน demand footprint ที่กว้าง ไม่ใช่ niche tech |
| Shared decks | ≥ 30 | คู่ต้องมีหลักฐานจริงจำนวนมากพอ ไม่ใช่เกิดจาก sample เล็ก |
| Jaccard | ≥ 0.30 | อย่างน้อย 30% ของ decklists ที่มี A หรือ B ต้องมีทั้งคู่ |
| Lift | ≥ 1.20 | ความสัมพันธ์ต้องสูงกว่าที่คาดจาก popularity ของแต่ละใบอย่างน้อย 20% |
| Shared archetypes | ≥ 3 | คู่ควรใช้ข้ามหลาย deck strategies ไม่ใช่เกิดจาก archetype เดียว |
| Mechanical/evolution pairs | ตัดรายการที่ประกาศล่วงหน้า | ป้องกันความสัมพันธ์ที่เป็นกติกาการสร้างเด็คโดยตรงมาชนะ ranking |

**Display Backbone ไม่ใช่ filter เชิงวิเคราะห์:** หลังคำนวณผลบน Analysis Graph แล้ว จึงเลือก Top watchlist nodes + Top pair candidates และลดเส้นเพื่อให้รูปอ่านง่ายใน A4


In [ ]:
# ============================================================
# MADT7202 — Pokémon TCG REAL Decklist Network v3
# TWO-GRAPH DESIGN:
#   Graph A = Pre-pruning co-occurrence network (evidence of raw complexity)
#   Graph B = Actionable Stock Network (business-aware signal refinement)
#
# REAL DATA ONLY:
# - Pulls public tournament data from Limitless official API
# - No dummy/fallback data
# - Stops if API/decklists cannot be validated
# ============================================================

from pathlib import Path
import sys, subprocess, importlib.util, time, json, re, itertools, os, shutil, math
from datetime import datetime, timezone

NEEDED = ["requests", "pandas", "numpy", "networkx", "matplotlib"]
missing = [p for p in NEEDED if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import requests
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 0) REPRODUCIBLE STUDY SETTINGS
# ------------------------------------------------------------
API = "https://play.limitlesstcg.com/api"
GAME = "PTCG"
ORGANIZER_ID = 36
TOURNAMENT_NAME_FILTER = "pumpkaweekly"
START = pd.Timestamp("2026-09-14T00:00:00Z")
END   = pd.Timestamp("2026-09-29T00:00:00Z")  # exclusive

# -------- Business-aware pruning thresholds --------
# WHY 10% prevalence?
# A weekly Core Watchlist should represent broad competitive usage, not niche tech.
MIN_CARD_PREVALENCE = 0.10   # 10% of valid decklists

# WHY 30 shared decks?
# A pair should have material support (~5% of a 584-deck corpus) before action.
MIN_SHARED_DECKS = 30

# WHY Jaccard >= 0.30?
# At least 30% of decks containing either card contain BOTH cards.
JACCARD_CUTOFF = 0.30

# WHY Lift >= 1.20?
# Pair co-occurrence must be at least 20% above independence expectation,
# so a pair is not selected only because both cards are individually popular.
MIN_LIFT = 1.20

# WHY >= 3 shared archetypes?
# The relationship should generalize across multiple deck strategies rather than
# be an artifact of one archetype.
MIN_SHARED_ARCHETYPES = 3

# Domain exclusion declared BEFORE ranking.
# These are obvious mechanical/evolution dependencies observed in the card pool.
# They are removed from Graph B because they are tautological deck-building links,
# not a novel inventory relationship. They remain visible in Graph A.
MECHANICAL_PAIRS = {
    tuple(sorted(x)) for x in [
        ("Dreepy", "Drakloak"),
        ("Dreepy", "Dragapult ex"),
        ("Drakloak", "Dragapult ex"),
        ("Dunsparce", "Dudunsparce"),
        ("Duskull", "Dusclops"),
        ("Duskull", "Dusknoir"),
        ("Dusclops", "Dusknoir"),
        ("Abra", "Kadabra"),
        ("Abra", "Alakazam"),
        ("Kadabra", "Alakazam"),
        ("Buneary", "Mega Lopunny ex"),
        ("N's Zorua", "N's Zoroark ex"),
        ("Applin", "Dipplin"),
        ("Riolu", "Mega Lucario ex"),
    ]
}

# Robustness check: hold all other criteria fixed and sweep Jaccard.
ROBUSTNESS_JACCARD = [0.25, 0.30, 0.35]

# Display graph is intentionally NOT the analysis graph.
# We compute metrics on the full business-filtered Analysis Graph first.
TOP_WATCHLIST_NODES_FOR_DISPLAY = 5
TOP_PAIR_EDGES_FOR_DISPLAY = 5
MAX_DISPLAY_EDGES_PER_NODE = 2

OUT = "pokemon_network_real_v3"
os.makedirs(OUT, exist_ok=True)

print("=" * 100)
print("REAL DATA ONLY | Limitless API | TWO-GRAPH Network v3")
print("Run UTC:", datetime.now(timezone.utc).isoformat())
print("=" * 100)

# ------------------------------------------------------------
# 1) API CLIENT
# ------------------------------------------------------------
S = requests.Session()
S.headers["User-Agent"] = "MADT7202-academic-network-analysis/3.0"

def get_json(path, params=None, retries=5):
    url = API + path
    last_err = None
    for i in range(retries):
        try:
            r = S.get(url, params=params, timeout=45)
            if r.status_code == 429:
                wait = int(r.headers.get("Retry-After", 10 * (i + 1)))
                print(f"HTTP 429 -> wait {wait}s")
                time.sleep(wait)
                continue
            r.raise_for_status()
            data = r.json()
            print(f"HTTP {r.status_code} | {r.url}")
            return data
        except Exception as e:
            last_err = e
            if i < retries - 1:
                time.sleep(2 ** i)
    raise RuntimeError(f"REAL API REQUEST FAILED: {url}\n{last_err}")

# ------------------------------------------------------------
# 2) DISCOVER REAL TOURNAMENTS
# ------------------------------------------------------------
tour = get_json("/tournaments", {
    "game": GAME, "organizerId": ORGANIZER_ID, "limit": 50
})

if not isinstance(tour, list) or not tour:
    raise RuntimeError("No tournaments returned. No fallback data.")

t = pd.DataFrame(tour)
required = {"id","game","format","name","date","players"}
if not required.issubset(t.columns):
    raise RuntimeError(f"Unexpected tournament schema: {t.columns.tolist()}")

t["date"] = pd.to_datetime(t["date"], utc=True)
sel = t[
    (t["game"] == GAME) &
    (t["date"] >= START) &
    (t["date"] < END) &
    t["name"].str.lower().str.contains(TOURNAMENT_NAME_FILTER, na=False)
].sort_values("date").copy()

if sel.empty:
    raise RuntimeError("No target tournaments found. No fallback data.")

if sel["format"].nunique() != 1:
    raise RuntimeError("Selected tournaments contain multiple formats.")

FORMAT_ID = sel["format"].iloc[0]

print("\n[1] REAL TOURNAMENTS SELECTED")
print(sel[["date","name","format","players","id"]].to_string(index=False))
print("Listed-player total:", int(sel["players"].sum()))
sel.to_csv(f"{OUT}/01_tournaments.csv", index=False)

# ------------------------------------------------------------
# 3) FETCH STANDINGS / DECKLISTS
# ------------------------------------------------------------
standings_all, source_audit = [], []

for _, row in sel.iterrows():
    tid = row["id"]
    details = get_json(f"/tournaments/{tid}/details")
    standings = get_json(f"/tournaments/{tid}/standings")

    if not isinstance(standings, list):
        raise RuntimeError(f"Standings is not a list for {tid}")

    print(
        f"  {row['date'].date()} | rows={len(standings)} | "
        f"decklists={details.get('decklists')} | public={details.get('isPublic')}"
    )

    source_audit.append({
        "tournament_id": tid,
        "tournament_name": row["name"],
        "date": row["date"].isoformat(),
        "format": row["format"],
        "listed_players": int(row["players"]),
        "standings_rows": len(standings),
        "decklists_setting": details.get("decklists"),
        "is_public": details.get("isPublic"),
        "public_page": f"https://play.limitlesstcg.com/tournament/{tid}/details",
        "api_standings": f"{API}/tournaments/{tid}/standings",
    })

    for s in standings:
        z = dict(s)
        z["_tid"] = tid
        z["_tname"] = row["name"]
        z["_tdate"] = row["date"].isoformat()
        standings_all.append(z)

    time.sleep(0.35)

pd.DataFrame(source_audit).to_csv(f"{OUT}/02_source_audit.csv", index=False)

# ------------------------------------------------------------
# 4) DECKLIST PARSER
# ------------------------------------------------------------
COUNT_KEYS = {"count","quantity","amount","copies","qty"}

def category_from(path):
    p = " ".join(map(str,path)).lower()
    if "pokemon" in p or "pokémon" in p: return "Pokemon"
    if "trainer" in p: return "Trainer"
    if "energy" in p: return "Energy"
    return "Unknown"

def parse_text_decklist(text):
    rows, cat = [], "Unknown"
    for raw in str(text).splitlines():
        line = raw.strip()
        low = line.lower()
        if not line: continue
        if low.startswith(("pokemon:","pokémon:")): cat="Pokemon"; continue
        if low.startswith("trainer:"): cat="Trainer"; continue
        if low.startswith("energy:"): cat="Energy"; continue
        if low.startswith("total cards"): continue
        m = re.match(r"^(\d+)\s+(.+)$", line)
        if not m: continue
        copies = int(m.group(1))
        rest = m.group(2).strip()
        name = re.sub(
            r"\s+[A-Z0-9-]{2,12}\s+\d+[A-Za-z-]*(?:\s+PH)?$",
            "", rest
        ).strip()
        rows.append((name or rest, copies, cat))
    return rows

def parse_decklist(obj):
    found = []
    if isinstance(obj, str):
        found.extend(parse_text_decklist(obj))
    else:
        def walk(x, path=()):
            if isinstance(x, dict):
                keymap = {str(k).lower(): k for k in x}
                copies = None
                for ck in COUNT_KEYS:
                    if ck in keymap:
                        try:
                            val = int(x[keymap[ck]])
                            if val > 0:
                                copies = val
                                break
                        except Exception:
                            pass

                name = None
                for nk in ("name","cardname","card_name"):
                    if nk in keymap and isinstance(x[keymap[nk]], str):
                        name = x[keymap[nk]].strip()
                        break

                if name is None and copies is not None:
                    card = x.get("card")
                    if isinstance(card, str):
                        name = card.strip()
                    elif isinstance(card, dict) and isinstance(card.get("name"), str):
                        name = card["name"].strip()

                if name and copies:
                    found.append((name, copies, category_from(path)))
                    return

                for k,v in x.items():
                    if isinstance(v, str) and str(k).lower() in {
                        "text","list","decktext","deck_text"
                    }:
                        found.extend(parse_text_decklist(v))
                    else:
                        walk(v, path+(k,))
            elif isinstance(x, list):
                for y in x:
                    walk(y, path)
            elif isinstance(x, str):
                p = parse_text_decklist(x)
                if p:
                    found.extend(p)
        walk(obj)

    if not found:
        return []

    d = pd.DataFrame(found, columns=["card","copies","category"])
    d["card"] = d["card"].astype(str).str.strip()
    d = d[(d["card"]!="") & (d["copies"]>0)].copy()

    g = d.groupby(["card","category"], as_index=False)["copies"].sum()
    bestcat = (
        g.sort_values("copies", ascending=False)
        .drop_duplicates("card")
        .set_index("card")["category"]
    )
    total = g.groupby("card", as_index=False)["copies"].sum()
    total["category"] = total["card"].map(bestcat)
    return total.to_dict("records")

deck_rows, card_rows = [], []

for s in standings_all:
    player = s.get("player")
    uid = f"{s['_tid']}::{player}"

    deck_obj = s.get("deck")
    deck_name = deck_obj.get("name") if isinstance(deck_obj,dict) else None
    deck_id = deck_obj.get("id") if isinstance(deck_obj,dict) else None

    payload = s.get("decklist")
    parsed = parse_decklist(payload) if payload is not None else []
    total_copies = int(sum(r["copies"] for r in parsed))

    deck_rows.append({
        "deck_uid":uid, "tournament_id":s["_tid"], "tournament":s["_tname"],
        "date":s["_tdate"], "player":player, "placing":s.get("placing"),
        "deck_id":deck_id, "deck_name":deck_name,
        "has_decklist":payload is not None,
        "unique_card_names":len(parsed),
        "total_copies":total_copies,
        "valid_60":total_copies==60
    })

    for r in parsed:
        card_rows.append({
            "deck_uid":uid, "deck_name":deck_name, "tournament_id":s["_tid"],
            "date":s["_tdate"], "player":player,
            "card":r["card"], "category":r["category"], "copies":r["copies"]
        })

decks = pd.DataFrame(deck_rows)
cards = pd.DataFrame(card_rows)

if cards.empty:
    raise RuntimeError("Parsed card rows = 0. Stopped.")

print("\n[2] DATA QUALITY")
print("Standings/player records :", len(decks))
print("Has decklist payload     :", int(decks["has_decklist"].sum()))
print("Valid 60-card decklists  :", int(decks["valid_60"].sum()))
print("Parsed card-line rows    :", len(cards))

valid_uids = set(decks.loc[decks["valid_60"],"deck_uid"])
decks = decks[decks["deck_uid"].isin(valid_uids)].copy()
cards = cards[cards["deck_uid"].isin(valid_uids)].copy()
N = len(decks)

if N == 0:
    raise RuntimeError("No valid 60-card decklists.")

decks.to_csv(f"{OUT}/03_valid_decklists.csv", index=False)
cards.to_csv(f"{OUT}/04_card_lines_used.csv", index=False)

# ------------------------------------------------------------
# 5) TECHNICAL CLEANING ONLY
# ------------------------------------------------------------
BASIC = {
    "grass energy","fire energy","water energy","lightning energy",
    "psychic energy","fighting energy","darkness energy",
    "metal energy","fairy energy"
}
def is_basic_energy(name):
    n = re.sub(r"\s+"," ",str(name).strip().lower())
    return n in BASIC or (n.startswith("basic ") and n.endswith(" energy")) or n.startswith("basic {")

cards_nb = cards[~cards["card"].map(is_basic_energy)].copy()

deck_card = (
    cards_nb.groupby(["deck_uid","deck_name","card"], as_index=False)["copies"].sum()
)
deck_card["present"] = 1

presence_all = (
    deck_card.pivot_table(
        index="deck_uid", columns="card", values="present",
        aggfunc="max", fill_value=0
    )
    .reindex(decks["deck_uid"], fill_value=0)
    .astype(np.uint8)
)

print("\n[3] TECHNICAL CLEANING")
print("Valid decklists used       :", N)
print("Unique non-Basic-Energy cards:", presence_all.shape[1])
print("Deck × card matrix         :", presence_all.shape)

# ------------------------------------------------------------
# 6) GRAPH A — PRE-PRUNING NETWORK
#    No business thresholds yet.
# ------------------------------------------------------------
X_all = presence_all.to_numpy(dtype=np.int32)
names_all = presence_all.columns.tolist()
co_all = X_all.T @ X_all

Graw = nx.Graph()
Graw.add_nodes_from(names_all)

raw_edges = []
for i in range(len(names_all)):
    for j in range(i+1, len(names_all)):
        shared = int(co_all[i,j])
        if shared >= 1:
            a,b = names_all[i], names_all[j]
            Graw.add_edge(a,b,shared_decks=shared)
            raw_edges.append((a,b,shared))

raw_edges_df = pd.DataFrame(raw_edges, columns=["card_a","card_b","shared_decks"])
raw_edges_df.to_csv(f"{OUT}/05_graphA_raw_edges.csv", index=False)

print("\n[4] GRAPH A — PRE-PRUNING")
print("Nodes:", Graw.number_of_nodes())
print("Edges:", Graw.number_of_edges())
print("Interpretation: this intentionally shows raw complexity before business pruning.")

# Fast layout for a large raw graph. No labels by design.
plt.figure(figsize=(14,10))
pos_raw = nx.spring_layout(Graw, seed=42, iterations=35, k=0.35)
raw_sizes = [
    8 + 180 * (presence_all[n].sum()/N)
    for n in Graw.nodes()
]
nx.draw_networkx_nodes(Graw, pos_raw, node_size=raw_sizes, alpha=0.55)
nx.draw_networkx_edges(Graw, pos_raw, width=0.35, alpha=0.035)
plt.title(
    "Graph A — Pre-pruning Card Co-occurrence Network\n"
    f"{N} real decklists | technical cleaning only | "
    f"{Graw.number_of_nodes()} nodes / {Graw.number_of_edges()} edges"
)
plt.axis("off")
plt.tight_layout()
plt.savefig(f"{OUT}/06_graphA_prepruning.png", dpi=220, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# 7) BUSINESS-AWARE SIGNAL REFINEMENT
# ------------------------------------------------------------
min_card_decks = math.ceil(MIN_CARD_PREVALENCE * N)
card_counts_all = presence_all.sum(axis=0).astype(int)
kept_cards = card_counts_all[card_counts_all >= min_card_decks].index.tolist()

presence = presence_all[kept_cards].copy()
X = presence.to_numpy(dtype=np.int32)
names = presence.columns.tolist()
co = X.T @ X

# deck_uid -> archetype/deck_name
arch_by_uid = (
    decks.set_index("deck_uid")["deck_name"]
    .fillna("Unclassified")
    .astype(str)
)
arch_arr = arch_by_uid.reindex(presence.index).to_numpy()

pair_rows = []
for i in range(len(names)):
    a = names[i]
    count_a = int(co[i,i])

    for j in range(i+1, len(names)):
        b = names[j]
        count_b = int(co[j,j])
        shared = int(co[i,j])
        if shared == 0:
            continue

        union = count_a + count_b - shared
        jaccard = shared / union

        p_a = count_a / N
        p_b = count_b / N
        p_ab = shared / N
        lift = p_ab / (p_a*p_b)

        both_mask = (X[:,i] == 1) & (X[:,j] == 1)
        shared_archetypes = len(set(arch_arr[both_mask]))

        key = tuple(sorted((a,b)))
        mechanical = key in MECHANICAL_PAIRS

        pair_rows.append({
            "card_a":a, "card_b":b,
            "count_a":count_a, "count_b":count_b,
            "shared_decks":shared,
            "jaccard":jaccard,
            "lift":lift,
            "shared_archetypes":shared_archetypes,
            "mechanical_pair":mechanical
        })

pairs = pd.DataFrame(pair_rows)
pairs.to_csv(f"{OUT}/07_pairs_after_card_prevalence.csv", index=False)

eligible = pairs[
    (pairs["shared_decks"] >= MIN_SHARED_DECKS) &
    (pairs["jaccard"] >= JACCARD_CUTOFF) &
    (pairs["lift"] >= MIN_LIFT) &
    (pairs["shared_archetypes"] >= MIN_SHARED_ARCHETYPES) &
    (~pairs["mechanical_pair"])
].copy()

eligible = eligible.sort_values(
    ["jaccard","shared_decks","lift","shared_archetypes"],
    ascending=[False,False,False,False]
).reset_index(drop=True)

if eligible.empty:
    raise RuntimeError(
        "No pair survived the declared business criteria. "
        "Do not silently relax thresholds; review the business assumptions."
    )

eligible.to_csv(f"{OUT}/08_graphB_eligible_edges.csv", index=False)

print("\n[5] SIGNAL REFINEMENT")
print(f"Card prevalence >= {MIN_CARD_PREVALENCE:.0%}: {len(names)} cards (minimum {min_card_decks} decks)")
print(f"Shared decks >= {MIN_SHARED_DECKS}")
print(f"Jaccard >= {JACCARD_CUTOFF:.2f}")
print(f"Lift >= {MIN_LIFT:.2f}")
print(f"Shared archetypes >= {MIN_SHARED_ARCHETYPES}")
print(f"Declared mechanical/evolution pairs excluded: {len(MECHANICAL_PAIRS)}")
print("Eligible business edges:", len(eligible))

print("\nTop 15 eligible relationships:")
print(
    eligible[
        ["card_a","card_b","shared_decks","jaccard","lift","shared_archetypes"]
    ].head(15).to_string(index=False, float_format=lambda x:f"{x:.3f}")
)

# ------------------------------------------------------------
# 8) ANALYSIS GRAPH — metrics are calculated HERE
# ------------------------------------------------------------
G = nx.Graph()

for r in eligible.itertuples(index=False):
    G.add_edge(
        r.card_a, r.card_b,
        shared_decks=int(r.shared_decks),
        jaccard=float(r.jaccard),
        lift=float(r.lift),
        shared_archetypes=int(r.shared_archetypes)
    )

bet = nx.betweenness_centrality(G, normalized=True, weight=None)
weighted_degree = {
    n: sum(d["jaccard"] for _,_,d in G.edges(n,data=True))
    for n in G.nodes()
}
degree = dict(G.degree())
prev = (presence.sum(axis=0)/N).to_dict()
deck_count = presence.sum(axis=0).astype(int).to_dict()

metrics = pd.DataFrame({
    "card":list(G.nodes()),
    "deck_count":[int(deck_count[n]) for n in G.nodes()],
    "prevalence":[float(prev[n]) for n in G.nodes()],
    "degree":[degree[n] for n in G.nodes()],
    "weighted_degree_jaccard":[weighted_degree[n] for n in G.nodes()],
    "betweenness":[bet[n] for n in G.nodes()]
}).sort_values(
    ["weighted_degree_jaccard","prevalence","betweenness"],
    ascending=False
).reset_index(drop=True)

metrics.to_csv(f"{OUT}/09_graphB_node_metrics_ANALYSIS_GRAPH.csv", index=False)

print("\n[6] ANALYSIS GRAPH")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Components:", nx.number_connected_components(G))
print("\nTop watchlist candidates:")
print(
    metrics.head(10).to_string(
        index=False,
        formatters={
            "prevalence":"{:.1%}".format,
            "weighted_degree_jaccard":"{:.3f}".format,
            "betweenness":"{:.3f}".format
        }
    )
)

# ------------------------------------------------------------
# 9) DISPLAY BACKBONE — visual simplification ONLY
#    This step does NOT change centrality results.
# ------------------------------------------------------------
top_watch = metrics.head(TOP_WATCHLIST_NODES_FOR_DISPLAY)["card"].tolist()
top_pairs = eligible.head(TOP_PAIR_EDGES_FOR_DISPLAY).copy()

display_nodes = set(top_watch)
for r in top_pairs.itertuples(index=False):
    display_nodes.add(r.card_a)
    display_nodes.add(r.card_b)

sub_edges = eligible[
    eligible["card_a"].isin(display_nodes) &
    eligible["card_b"].isin(display_nodes)
].copy()

# If dense, keep each node's top 2 strongest relationships for display only.
display_keep = set()
for n in display_nodes:
    s = sub_edges[
        (sub_edges["card_a"]==n) | (sub_edges["card_b"]==n)
    ].sort_values(
        ["jaccard","shared_decks","lift"],
        ascending=[False,False,False]
    ).head(MAX_DISPLAY_EDGES_PER_NODE)

    for r in s.itertuples(index=False):
        display_keep.add(tuple(sorted((r.card_a,r.card_b))))

display_edges = sub_edges[
    sub_edges.apply(
        lambda r: tuple(sorted((r["card_a"],r["card_b"]))) in display_keep,
        axis=1
    )
].copy()

Gdisp = nx.Graph()
for r in display_edges.itertuples(index=False):
    Gdisp.add_edge(
        r.card_a,r.card_b,
        jaccard=float(r.jaccard),
        shared_decks=int(r.shared_decks),
        lift=float(r.lift),
        shared_archetypes=int(r.shared_archetypes)
    )

display_edges.to_csv(f"{OUT}/10_graphB_display_edges.csv", index=False)

print("\n[7] DISPLAY BACKBONE")
print("Display nodes:", Gdisp.number_of_nodes())
print("Display edges:", Gdisp.number_of_edges())
print("NOTE: display pruning is for readability only; metrics came from the Analysis Graph.")

# Draw polished Graph B
plt.figure(figsize=(14,9))
pos = nx.spring_layout(Gdisp, seed=42, weight="jaccard", k=1.4, iterations=100)

node_sizes = []
for n in Gdisp.nodes():
    p = float(prev.get(n,0))
    node_sizes.append(900 + 3500*p)

edge_widths = [
    1.5 + 7*Gdisp[u][v]["jaccard"]
    for u,v in Gdisp.edges()
]

nx.draw_networkx_nodes(Gdisp,pos,node_size=node_sizes,alpha=0.92)
nx.draw_networkx_edges(Gdisp,pos,width=edge_widths,alpha=0.42)
nx.draw_networkx_labels(Gdisp,pos,font_size=9)

edge_labels = {
    (u,v): f"J={d['jaccard']:.2f}\nN={d['shared_decks']}"
    for u,v,d in Gdisp.edges(data=True)
}
nx.draw_networkx_edge_labels(
    Gdisp,pos,edge_labels=edge_labels,font_size=7,rotate=False
)

plt.title(
    "Graph B — Actionable Stock Network\n"
    f"Business-filtered Analysis Graph: {G.number_of_nodes()} nodes / {G.number_of_edges()} edges | "
    f"Display backbone: {Gdisp.number_of_nodes()} nodes / {Gdisp.number_of_edges()} edges"
)
plt.axis("off")
plt.tight_layout()
plt.savefig(f"{OUT}/11_graphB_actionable.png", dpi=240, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# 10) CONSTRUCTION CHOICE: RAW COUNT vs BUSINESS FILTER
# ------------------------------------------------------------
raw_count_top = pairs.sort_values(
    ["shared_decks","jaccard"], ascending=[False,False]
).head(10)

print("\n[8] CONSTRUCTION CHOICE CHECK")
print("Top by raw co-occurrence count:")
print(
    raw_count_top[
        ["card_a","card_b","shared_decks","jaccard","lift","shared_archetypes"]
    ].head(5).to_string(index=False, float_format=lambda x:f"{x:.3f}")
)

print("\nTop after business-aware refinement:")
print(
    eligible[
        ["card_a","card_b","shared_decks","jaccard","lift","shared_archetypes"]
    ].head(5).to_string(index=False, float_format=lambda x:f"{x:.3f}")
)

# ------------------------------------------------------------
# 11) ROBUSTNESS
# ------------------------------------------------------------
best = eligible.iloc[0]
best_key = tuple(sorted((best["card_a"],best["card_b"])))

rob = []
for cut in ROBUSTNESS_JACCARD:
    ed = pairs[
        (pairs["shared_decks"] >= MIN_SHARED_DECKS) &
        (pairs["jaccard"] >= cut) &
        (pairs["lift"] >= MIN_LIFT) &
        (pairs["shared_archetypes"] >= MIN_SHARED_ARCHETYPES) &
        (~pairs["mechanical_pair"])
    ].copy()

    gg = nx.Graph()
    for r in ed.itertuples(index=False):
        gg.add_edge(r.card_a,r.card_b)

    survives = any(
        tuple(sorted((r.card_a,r.card_b))) == best_key
        for r in ed.itertuples(index=False)
    )

    rob.append({
        "jaccard_cutoff":cut,
        "nodes":gg.number_of_nodes(),
        "edges":gg.number_of_edges(),
        "components":nx.number_connected_components(gg) if gg.number_of_nodes() else 0,
        "top_pair_survives":survives
    })

robustness = pd.DataFrame(rob)
robustness.to_csv(f"{OUT}/12_robustness.csv", index=False)

print("\n[9] ROBUSTNESS")
print(robustness.to_string(index=False))

# ------------------------------------------------------------
# 12) A4-READY SUMMARY TABLE
# ------------------------------------------------------------
watch = metrics.iloc[0]

a4 = pd.DataFrame([
    {
        "Finding":"Graph reduction",
        "Evidence":(
            f"Graph A {Graw.number_of_nodes()} nodes/{Graw.number_of_edges()} edges -> "
            f"Analysis Graph {G.number_of_nodes()} nodes/{G.number_of_edges()} edges -> "
            f"Display Graph {Gdisp.number_of_nodes()} nodes/{Gdisp.number_of_edges()} edges"
        )
    },
    {
        "Finding":"Core Watchlist",
        "Evidence":(
            f"{watch['card']}: prevalence={watch['prevalence']:.1%}; "
            f"weighted degree={watch['weighted_degree_jaccard']:.3f}; "
            f"betweenness={watch['betweenness']:.3f}"
        )
    },
    {
        "Finding":"Pair-check",
        "Evidence":(
            f"{best['card_a']} — {best['card_b']}: shared={int(best['shared_decks'])}; "
            f"Jaccard={best['jaccard']:.3f}; Lift={best['lift']:.3f}; "
            f"breadth={int(best['shared_archetypes'])} archetypes"
        )
    },
    {
        "Finding":"Robustness",
        "Evidence":"; ".join(
            f"J={r.jaccard_cutoff:.2f}:{'Y' if r.top_pair_survives else 'N'}"
            for r in robustness.itertuples(index=False)
        )
    }
])

a4.to_csv(f"{OUT}/13_A4_findings.csv", index=False)
print("\n[10] A4-READY FINDINGS")
print(a4.to_string(index=False))

# ------------------------------------------------------------
# 13) AUDIT SUMMARY + ZIP
# ------------------------------------------------------------
summary = [
    "MADT7202 REAL DATA AUDIT SUMMARY — v3",
    f"Run UTC: {datetime.now(timezone.utc).isoformat()}",
    "",
    "BUSINESS QUESTION",
    "Which cards should be monitored as Core Watchlist stock, and which card pairs",
    "should be checked together before a purchase order?",
    "",
    "DATA",
    f"Period: {START.date()} to {(END-pd.Timedelta(days=1)).date()}",
    f"Tournaments: {len(sel)}",
    f"Valid decklists: {N}",
    f"Parsed card-line rows: {len(cards)}",
    f"Unique non-Basic-Energy cards: {presence_all.shape[1]}",
    "",
    "GRAPH A — PRE-PRUNING",
    f"Nodes: {Graw.number_of_nodes()}",
    f"Edges: {Graw.number_of_edges()}",
    "",
    "SIGNAL REFINEMENT",
    f"Card prevalence >= {MIN_CARD_PREVALENCE:.0%} ({min_card_decks} decks)",
    f"Shared decks >= {MIN_SHARED_DECKS}",
    f"Jaccard >= {JACCARD_CUTOFF:.2f}",
    f"Lift >= {MIN_LIFT:.2f}",
    f"Shared archetypes >= {MIN_SHARED_ARCHETYPES}",
    f"Mechanical pairs declared/excluded: {len(MECHANICAL_PAIRS)}",
    "",
    "GRAPH B — ANALYSIS GRAPH",
    f"Nodes: {G.number_of_nodes()}",
    f"Edges: {G.number_of_edges()}",
    f"Components: {nx.number_connected_components(G)}",
    "",
    "GRAPH B — DISPLAY BACKBONE",
    f"Nodes: {Gdisp.number_of_nodes()}",
    f"Edges: {Gdisp.number_of_edges()}",
    "",
    "TOP BUSINESS FINDINGS",
    f"Watchlist: {watch['card']} | prevalence={watch['prevalence']:.6f} | "
    f"weighted_degree={watch['weighted_degree_jaccard']:.6f} | betweenness={watch['betweenness']:.6f}",
    f"Pair: {best['card_a']} — {best['card_b']} | shared={int(best['shared_decks'])} | "
    f"Jaccard={best['jaccard']:.6f} | lift={best['lift']:.6f} | "
    f"breadth={int(best['shared_archetypes'])}",
    "",
    "LIMITATION",
    "Tournament co-occurrence is a proxy for competitive usage, not proof of customer co-purchase.",
    "Validate the recommended pair against store POS before bundle/cross-sell."
]

for s in source_audit:
    summary.extend([s["public_page"], s["api_standings"]])

summary = "\n".join(summary)
Path(f"{OUT}/14_AUDIT_SUMMARY.txt").write_text(summary, encoding="utf-8")

zip_path = shutil.make_archive("pokemon_network_real_v3_evidence","zip",OUT)

print("\n"+"="*100)
print(summary)
print("="*100)
print("Evidence ZIP:", zip_path)
print("IMPORTANT: use ONLY numbers printed/exported by this run in the final A4.")
